# KIR Project

## Allele curation: 3-digit protein groups

KIR allele names are hierarchical: in `KIR2DL1*0010101` the first **3** digits identify the
distinct protein, digits 4-5 are synonymous coding changes and digits 6-7 are non-coding
differences. Collapsing to the 3-digit group therefore removes all redundancy at the protein
level without comparing a single residue - we verify that below rather than assume it.

In [11]:
import re
import subprocess
from pathlib import Path

import pandas as pd
from Bio import SeqIO

PROJ = Path("/Users/roessner/Documents/PostDoc/Data/261105_kir_project")
ALLELE_DIR = PROJ / "kir_alleles"
OUT_DIR = PROJ / "alignments"
MUSCLE = "/Users/roessner/miniconda3/envs/phd/bin/muscle"

OUT_DIR.mkdir(exist_ok=True)

### 1. Parse every per-gene protein FASTA

`kir_prot.fasta` is the all-gene concatenation and `KIR2DL5_prot.fasta` is exactly the union of
`KIR2DL5A` + `KIR2DL5B`, so both are skipped to avoid duplicating entries. The assertion catches
it if that ever changes.

A sequence counts as `complete` only if it reaches its gene's full length **and** does not end in
`X`. IPD-KIR holds many partial submissions, and `X` marks a translation that ran into an
undetermined final codon.

In [12]:
SKIP_FILES = {"kir_prot.fasta", "KIR2DL5_prot.fasta"}
HEADER_RE = re.compile(r"^(?P<acc>KIR:\w+)\s+(?P<gene>[A-Z0-9]+)\*(?P<digits>\d+)")


def load_gene(path):
    """One row per allele entry in a *_prot.fasta file."""
    rows = []
    for rec in SeqIO.parse(path, "fasta"):
        m = HEADER_RE.match(rec.description)
        if m is None:
            raise ValueError(f"unparsed header in {path.name}: {rec.description!r}")
        digits = m.group("digits")
        rows.append({
            "gene": m.group("gene"),
            "accession": m.group("acc"),
            "allele": f"{m.group('gene')}*{digits}",
            "digits": digits,
            "group3": digits[:3],
            "seq": str(rec.seq),
            "length": len(rec.seq),
            "has_X": "X" in str(rec.seq),
        })
    return pd.DataFrame(rows)


prot_files = sorted(p for p in ALLELE_DIR.glob("*_prot.fasta") if p.name not in SKIP_FILES)
alleles = pd.concat([load_gene(p) for p in prot_files], ignore_index=True)

n_dup = alleles.accession.duplicated().sum()
assert n_dup == 0, f"{n_dup} duplicated accessions - check SKIP_FILES"


# X is IPD-KIR's placeholder for an undetermined residue. In these files it only ever
# occurs as the final character, marking a translation truncated at the C-terminus -
# asserted below so this fails loudly if a future release changes that.
alleles["x_terminated"] = alleles["seq"].str.endswith("X")

x_rows = alleles[alleles.has_X]
assert (x_rows.seq.str.count("X") == 1).all(), "sequence with more than one X"
assert x_rows.x_terminated.all(), "X occurs at a non-terminal position"

# A gene's full length is the modal length among UNtruncated sequences. Using the
# overall mode would call KIR3DS1's 383-residue truncations complete, since they
# outnumber its one genuinely full-length entry 35 to 1.
untruncated = alleles[~alleles.x_terminated]
full_len = untruncated.groupby("gene")["length"].agg(lambda s: s.mode().iat[0])
assert full_len.notna().all(), "a gene has no untruncated sequence to measure"

alleles["full_length"] = alleles["gene"].map(full_len)
alleles["complete"] = (alleles.length == alleles.full_length) & ~alleles.x_terminated

print(f"{len(alleles)} allele entries across {alleles.gene.nunique()} genes")
print(f"{int(alleles.complete.sum())} complete, {int(alleles.x_terminated.sum())} X-terminated")
alleles.groupby("gene").agg(entries=("allele", "size"), groups=("group3", "nunique"))

1983 allele entries across 15 genes
1706 complete, 189 X-terminated


,entries,groups
gene,,
KIR2DL1,433,131
KIR2DL2,35,16
KIR2DL3,70,38
KIR2DL4,118,61
KIR2DL5A,45,21
KIR2DL5B,49,23
KIR2DS1,62,20
KIR2DS2,65,22
KIR2DS3,71,24


### 2. Collapse to one representative per 3-digit group

Representative choice is deterministic: a complete sequence beats a partial one, then the longest,
then the lowest allele number (`*0010101` is the first-described allele by convention).

In [13]:
def pick_representative(g):
    """Complete over partial, then untruncated, then longest, then lowest allele number."""
    g = g.sort_values(
        ["complete", "x_terminated", "length", "digits"],
        ascending=[False, True, False, True],
    )
    return g.iloc[0]


reps = (
    alleles.groupby(["gene", "group3"], as_index=False, group_keys=False)
    .apply(pick_representative)
    .reset_index(drop=True)
)

print(f"{len(reps)} group representatives from {len(alleles)} allele entries")
reps[["gene", "group3", "allele", "length", "complete", "x_terminated"]].head()

868 group representatives from 1983 allele entries


/var/folders/mg/jhh2rvf11vd1jd013_6qqg200000gn/T/ipykernel_65722/1078323801.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_representative)


,gene,group3,allele,length,complete,x_terminated
0,KIR2DL1,001,KIR2DL1*0010101,348,True,False
1,KIR2DL1,002,KIR2DL1*0020101,348,True,False
2,KIR2DL1,003,KIR2DL1*00301,348,True,False
3,KIR2DL1,004,KIR2DL1*0040101,348,True,False
4,KIR2DL1,005,KIR2DL1*005,334,False,False


### 3. Audit the collapse

Two things worth knowing before trusting the reduction: whether any group contains two genuinely
different full-length proteins (it should not, by the naming scheme), and which groups have no
complete member at all, so their representative is a partial sequence.

In [14]:
audit = (
    alleles.groupby(["gene", "group3"])
    .apply(
        lambda g: pd.Series({
            "n_entries": len(g),
            "n_complete": int(g.complete.sum()),
            "n_distinct_complete_seqs": g.loc[g.complete, "seq"].nunique(),
            "members": ";".join(sorted(g.allele)),
        }),
        include_groups=False,
    )
    .reset_index()
)

conflicting = audit[audit.n_distinct_complete_seqs > 1]
partial_only = audit[audit.n_complete == 0]

print(f"groups with conflicting complete sequences: {len(conflicting)}")
print(f"partial-only groups (representative is incomplete): {len(partial_only)}")
partial_only.groupby("gene").size().rename("partial_only_groups").to_frame()

groups with conflicting complete sequences: 0
partial-only groups (representative is incomplete): 158


,partial_only_groups
gene,
KIR2DL1,16
KIR2DL2,1
KIR2DL3,4
KIR2DL4,28
KIR2DL5A,1
KIR2DL5B,2
KIR2DS2,2
KIR2DS3,4
KIR2DS4,12


### 4. Align each gene separately

One MSA per gene over its group representatives. FASTA ids are `GENE*group|representative_allele`
so the group is readable straight off the alignment, and the aligned output is re-sorted back into
group order because MUSCLE writes guide-tree order instead.

In [15]:
def align_gene(gene, rep_df, out_dir=OUT_DIR, muscle=MUSCLE):
    """Write the group representatives of one gene and align them with MUSCLE."""
    sub = rep_df[rep_df.gene == gene].sort_values("group3")
    in_fa = out_dir / f"{gene}_groups.fasta"
    out_fa = out_dir / f"{gene}_groups_aln.fasta"

    order = [f"{r.gene}*{r.group3}|{r.allele}" for _, r in sub.iterrows()]
    with in_fa.open("w") as fh:
        for seq_id, seq in zip(order, sub.seq):
            fh.write(f">{seq_id}\n{seq}\n")

    if len(sub) < 2:  # MUSCLE refuses a single sequence
        out_fa.write_text(in_fa.read_text())
        return out_fa

    subprocess.run(
        [muscle, "-align", str(in_fa), "-output", str(out_fa)],
        check=True, capture_output=True,
    )

    # MUSCLE emits sequences in guide-tree order, which is arbitrary and shifts when
    # the input changes. Restore 3-digit group order so the file is predictable.
    recs = {r.id: r for r in SeqIO.parse(out_fa, "fasta")}
    assert set(recs) == set(order), f"{gene}: MUSCLE changed the sequence set"
    SeqIO.write([recs[i] for i in order], out_fa, "fasta")
    return out_fa


genes = sorted(reps.gene.unique())
aln_paths = {g: align_gene(g, reps) for g in genes}  # ~30 s for all 15 genes
print(f"wrote {len(aln_paths)} alignments to {OUT_DIR}")

wrote 15 alignments to /Users/roessner/Documents/PostDoc/Data/261105_kir_project/alignments


### 5. Alignment matrices and variable positions

Each alignment becomes a (group x column) character matrix. A column counts as variable only if
the **non-gap** residues differ, so partial sequences do not inflate the polymorphism count.

In [16]:
def alignment_matrix(path):
    """Alignment as a DataFrame: rows = sequences, columns = alignment positions.

    Rows are sorted by 3-digit group, so the index is stable regardless of the order
    the aligner happened to write - never rely on positional row order from MUSCLE.
    """
    recs = list(SeqIO.parse(path, "fasta"))
    mat = pd.DataFrame([list(str(r.seq)) for r in recs], index=[r.id for r in recs])
    return mat.sort_index(key=lambda ix: ix.str.split("|").str[0].str.split("*").str[1])


matrices, var_tables, summary = {}, {}, []
for gene, path in aln_paths.items():
    mat = alignment_matrix(path)
    n_resid = mat.apply(lambda col: col[col != "-"].nunique())
    matrices[gene] = mat
    var_tables[gene] = mat.loc[:, n_resid > 1]
    summary.append({
        "gene": gene,
        "n_groups": mat.shape[0],
        "aln_width": mat.shape[1],
        "n_variable_cols": int((n_resid > 1).sum()),
    })

summary = pd.DataFrame(summary).sort_values("gene").reset_index(drop=True)
summary

,gene,n_groups,aln_width,n_variable_cols
0,KIR2DL1,131,348,125
1,KIR2DL2,16,348,21
2,KIR2DL3,38,350,59
3,KIR2DL4,61,377,71
4,KIR2DL5A,21,375,19
5,KIR2DL5B,23,375,19
6,KIR2DS1,20,304,17
7,KIR2DS2,22,304,27
8,KIR2DS3,24,304,22
9,KIR2DS4,23,305,87


### 6. Save the curated tables

`kir_group_representatives.csv` is the auditable curation record: one row per retained group with
the member alleles it stands for. The alignments themselves are in `alignments/`.

In [17]:
reps_out = reps.merge(
    audit[["gene", "group3", "n_entries", "n_complete", "members"]],
    on=["gene", "group3"], how="left",
)
reps_out = reps_out[[
    "gene", "group3", "allele", "accession", "length", "full_length",
    "complete", "has_X", "x_terminated", "n_entries", "n_complete", "members", "seq",
]]

reps_out.to_csv(PROJ / "kir_group_representatives.csv", index=False)
audit.to_csv(PROJ / "kir_group_audit.csv", index=False)
summary.to_csv(PROJ / "kir_alignment_summary.csv", index=False)

# group3 is a 3-character label, not a number: read it back with dtype={"group3": str}
# or 001 silently becomes 1 and joins against the alignment ids fail.
print(f"saved {len(reps_out)} representatives")
reps_out.drop(columns="seq").head()

saved 868 representatives


,gene,group3,allele,accession,length,full_length,complete,has_X,x_terminated,n_entries,n_complete,members
0,KIR2DL1,001,KIR2DL1*0010101,KIR:KIR00001,348,348,True,False,False,11,11,KIR2DL1*0010101;KIR2DL1*0010102;KIR2DL1*001010...
1,KIR2DL1,002,KIR2DL1*0020101,KIR:KIR00002,348,348,True,False,False,52,52,KIR2DL1*0020101;KIR2DL1*0020102;KIR2DL1*002010...
2,KIR2DL1,003,KIR2DL1*00301,KIR:KIR00003,348,348,True,False,False,130,130,KIR2DL1*00301;KIR2DL1*0030201;KIR2DL1*0030202;...
3,KIR2DL1,004,KIR2DL1*0040101,KIR:KIR00006,348,348,True,False,False,49,48,KIR2DL1*0040101;KIR2DL1*0040102;KIR2DL1*004010...
4,KIR2DL1,005,KIR2DL1*005,KIR:KIR00008,334,348,False,False,False,1,0,KIR2DL1*005


### Inspect a single gene

`var_tables[gene]` holds only the polymorphic columns, which is usually what you want to eyeball.
Column labels are 0-based alignment positions.

In [18]:
gene = "KIR2DL1"
print(f"{gene}: {matrices[gene].shape[0]} groups, "
      f"{matrices[gene].shape[1]} columns, {var_tables[gene].shape[1]} variable")

# rows are group-sorted, so this really is *001 onwards
var_tables[gene].iloc[:15, :25]

KIR2DL1: 131 groups, 348 columns, 125 variable


,3,4,27,28,29,30,31,35,36,40,...,49,50,51,52,53,54,55,56,62,63
KIR2DL1*001|KIR2DL1*0010101,L,V,K,P,S,L,L,G,P,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*002|KIR2DL1*0020101,L,F,K,P,S,L,L,G,P,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*003|KIR2DL1*00301,L,V,K,P,S,L,L,G,R,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*004|KIR2DL1*0040101,L,V,K,P,S,L,L,G,R,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*005|KIR2DL1*005,-,-,K,P,S,L,L,G,R,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*006|KIR2DL1*00601,L,V,K,P,S,L,L,G,R,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*007|KIR2DL1*0070101,L,V,K,P,S,L,L,G,R,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*008|KIR2DL1*0080101,L,F,K,P,S,L,L,D,P,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*009|KIR2DL1*009,-,V,K,P,S,L,L,G,R,S,...,W,S,D,V,M,F,E,H,E,G
KIR2DL1*010|KIR2DL1*010,L,V,K,P,S,L,L,G,R,S,...,W,S,D,V,M,F,E,H,E,G


## Ectodomain trimming

The IPD-KIR protein entries are full-length precursors: signal peptide, Ig-like domains, stem,
transmembrane helix and cytoplasmic tail. For ligand binding only the extracellular part is
informative, and most of the length difference between an inhibitory (`*DL*`) and an activating
(`*DS*`) gene sits in the cytoplasmic tail rather than at the binding end.

This section trims every sequence to its ectodomain - **all** Ig domains, so D1+D2 for the 2D
genes, D0+D2 for KIR2DL4/2DL5 and D0+D1+D2 for the 3D genes - and re-aligns per gene.

Boundaries are not hardcoded per gene. The transmembrane helix is located by hydrophobicity, and
the signal-peptide cleavage site is transferred from a single literature anchor (UniProt P43626,
KIR2DL1 signal peptide = residues 1-21) onto the other 14 genes through a cross-gene alignment.

In [ ]:
KD = dict(A=1.8, R=-4.5, N=-3.5, D=-3.5, C=2.5, Q=-3.5, E=-3.5, G=-0.4, H=-3.2,
          I=4.5, L=3.8, K=-3.9, M=1.9, F=2.8, P=-1.6, S=-0.8, T=-0.7, W=-0.9,
          Y=-1.3, V=4.2, X=0.0)

ANCHOR_GENE = "KIR2DL1"
ANCHOR_SIGNAL_END = 21  # UniProt P43626: mature chain starts at H22 (HEGVHRKP...)


def tm_start(seq, w=19):
    """1-based start of the most hydrophobic window, i.e. the TM helix."""
    best = (-99, None)
    for i in range(len(seq) - w):
        h = sum(KD[c] for c in seq[i:i + w]) / w
        if h > best[0]:
            best = (h, i)
    return best[1] + 1


def res_to_col(aln_seq, res_idx):
    """1-based residue number -> 0-based alignment column."""
    n = 0
    for col, ch in enumerate(aln_seq):
        if ch != "-":
            n += 1
            if n == res_idx:
                return col
    raise ValueError(f"residue {res_idx} beyond sequence")


def col_to_res(aln_seq, col):
    """0-based alignment column -> 1-based residue number at or before it."""
    return sum(1 for ch in aln_seq[:col + 1] if ch != "-")

### 7. Per-gene ectodomain boundaries

One complete, well-sampled reference per gene. The check to watch is the printed mature
N-terminus: every gene should begin on the same conserved motif (`HEGVHRKP` / `HVGGQDKP`), which
is what shows the transferred cleavage site landed in the right place - including KIR2DL4, whose
signal peptide is two residues longer than the rest.

In [ ]:
gene_ref = (reps_out[reps_out.complete]
            .sort_values("n_entries", ascending=False)
            .groupby("gene").head(1)
            .set_index("gene").sort_index())
assert gene_ref.complete.all(), "a gene reference is not a complete sequence"

cross_in, cross_out = OUT_DIR / "gene_refs.fasta", OUT_DIR / "gene_refs_aln.fasta"
with cross_in.open("w") as fh:
    for gene, r in gene_ref.iterrows():
        fh.write(f">{gene}\n{r.seq}\n")
subprocess.run([MUSCLE, "-align", str(cross_in), "-output", str(cross_out)],
               check=True, capture_output=True)
cross = {r.id: str(r.seq) for r in SeqIO.parse(cross_out, "fasta")}

anchor_col = res_to_col(cross[ANCHOR_GENE], ANCHOR_SIGNAL_END)

rows = []
for gene, r in gene_ref.iterrows():
    sig_end = col_to_res(cross[gene], anchor_col)
    tm = tm_start(r.seq)
    rows.append({
        "gene": gene, "ref_allele": r.allele, "ref_len": r.length,
        "signal_end": sig_end, "ecto_start": sig_end + 1,
        "ecto_end": tm - 1, "tm_start": tm, "ecto_len": tm - 1 - sig_end,
        "n_ig_domains": 3 if gene.startswith(("KIR3DL", "KIR3DS")) else 2,
    })

bounds = pd.DataFrame(rows).set_index("gene")
bounds["len_per_domain"] = (bounds.ecto_len / bounds.n_ig_domains).round(1)

for gene, r in gene_ref.iterrows():
    sig = bounds.loc[gene, "signal_end"]
    print(f"  {gene:9s} signal 1-{sig:<3d} mature starts {r.seq[sig:sig + 12]}")

bounds[["ref_allele", "signal_end", "ecto_start", "ecto_end", "tm_start",
        "ecto_len", "n_ig_domains", "len_per_domain"]]

### 8. Trim each representative to its ectodomain

Trimming by absolute residue number would be wrong for the partial sequences, whose coordinates
are shifted. Instead the reference boundaries are mapped onto the existing full-length alignment
columns and every sequence is cut there, so each one contributes exactly the region that is
homologous to the reference ectodomain.

In [ ]:
ecto_rows = []
for gene in bounds.index:
    b = bounds.loc[gene]
    aln = {r.id: str(r.seq) for r in SeqIO.parse(OUT_DIR / f"{gene}_groups_aln.fasta", "fasta")}
    ref_id = next(i for i in aln if i.endswith("|" + b.ref_allele))

    c0 = res_to_col(aln[ref_id], b.ecto_start)
    c1 = res_to_col(aln[ref_id], b.ecto_end)

    for seq_id, aln_seq in aln.items():
        frag = aln_seq[c0:c1 + 1].replace("-", "")
        ecto_rows.append({
            "gene": gene,
            "group3": seq_id.split("|")[0].split("*")[1],
            "allele": seq_id.split("|")[1],
            "ecto_seq": frag,
            "ecto_obs_len": len(frag),
            "ecto_ref_len": int(b.ecto_len),
            "ecto_full": len(frag) == int(b.ecto_len) and "X" not in frag,
        })

ecto = pd.DataFrame(ecto_rows)
ecto["ecto_cov"] = (ecto.ecto_obs_len / ecto.ecto_ref_len).round(3)

print(f"full ectodomain      : {int(ecto.ecto_full.sum())}")
print(f"partial ectodomain   : {int((~ecto.ecto_full & (ecto.ecto_obs_len > 0)).sum())}")
print(f"no ectodomain at all : {int((ecto.ecto_obs_len == 0).sum())}")
ecto.head()

### 9. Collapse identical ectodomains

Groups that differ only outside the ectodomain - in the stem, TM or cytoplasmic tail - become one
entry here. Only sequences with a **full** ectodomain take part: a fragment that happens to match
over its shorter span is not evidence of identity, so partials are kept aside rather than merged.

In [ ]:
ecto_full = ecto[ecto.ecto_full].copy()

# gene-prefixed so the id is unique across the whole table, not just within a gene -
# a bare per-gene integer would silently merge different genes' ectodomains on a join
ecto_full["ecto_id"] = (
    ecto_full.gene + "_e"
    + ecto_full.groupby("gene")["ecto_seq"]
    .transform(lambda s: pd.factorize(s)[0]).astype(str).str.zfill(3)
)

collapse = (ecto_full.groupby("gene")
            .agg(groups_in=("group3", "nunique"), ecto_unique=("ecto_seq", "nunique"))
            .assign(collapsed=lambda d: d.groups_in - d.ecto_unique))

print(f"{collapse.groups_in.sum()} full-ectodomain groups -> "
      f"{collapse.ecto_unique.sum()} unique ectodomains "
      f"({collapse.collapsed.sum()} collapsed)")
collapse

### 10. Re-align the unique ectodomains

One MSA per gene over the distinct ectodomains, written in group order as before. The lowest group
number is kept as the representative of each distinct ectodomain.

In [ ]:
ecto_uniq = (ecto_full.sort_values(["gene", "group3"])
             .drop_duplicates(["gene", "ecto_seq"]))

ecto_aln_paths = {}
for gene, sub in ecto_uniq.groupby("gene"):
    fa = OUT_DIR / f"{gene}_ecto.fasta"
    aln_fa = OUT_DIR / f"{gene}_ecto_aln.fasta"
    order = [f"{gene}*{r.group3}|{r.allele}" for r in sub.itertuples()]

    with fa.open("w") as fh:
        for sid, s in zip(order, sub.ecto_seq):
            fh.write(f">{sid}\n{s}\n")

    if len(sub) < 2:
        aln_fa.write_text(fa.read_text())
    else:
        subprocess.run([MUSCLE, "-align", str(fa), "-output", str(aln_fa)],
                       check=True, capture_output=True)
        recs = {r.id: r for r in SeqIO.parse(aln_fa, "fasta")}
        assert set(recs) == set(order), f"{gene}: MUSCLE changed the sequence set"
        SeqIO.write([recs[i] for i in order], aln_fa, "fasta")

    ecto_aln_paths[gene] = aln_fa

# one combined file of every unique ectodomain, unaligned - the genes differ in
# length and domain architecture, so a single cross-gene MSA is not meaningful
all_ecto = OUT_DIR / "all_ecto.fasta"
with all_ecto.open("w") as fh:
    for r in ecto_uniq.sort_values(["gene", "group3"]).itertuples():
        fh.write(f">{r.gene}*{r.group3}|{r.allele}\n{r.ecto_seq}\n")
print(f"combined: {all_ecto.name} with {len(ecto_uniq)} unique ectodomains")

ecto_matrices, ecto_var, ecto_summary = {}, {}, []
for gene, path in ecto_aln_paths.items():
    mat = alignment_matrix(path)
    n_resid = mat.apply(lambda col: col[col != "-"].nunique())
    ecto_matrices[gene] = mat
    ecto_var[gene] = mat.loc[:, n_resid > 1]
    ecto_summary.append({
        "gene": gene, "n_ecto": mat.shape[0], "aln_width": mat.shape[1],
        "n_variable_cols": int((n_resid > 1).sum()),
    })

ecto_summary = pd.DataFrame(ecto_summary).sort_values("gene").reset_index(drop=True)
ecto_summary

### 11. Save the ectodomain tables

In [ ]:
ecto_out = ecto.merge(bounds[["ref_allele", "signal_end", "tm_start", "n_ig_domains"]],
                      on="gene", how="left")
ecto_out = ecto_out.merge(ecto_full[["gene", "group3", "ecto_id"]],
                          on=["gene", "group3"], how="left")

ecto_out.to_csv(PROJ / "kir_ectodomains.csv", index=False)
bounds.to_csv(PROJ / "kir_ectodomain_bounds.csv")
ecto_summary.to_csv(PROJ / "kir_ectodomain_alignment_summary.csv", index=False)

print(f"saved {len(ecto_out)} rows; {ecto_out.ecto_id.notna().sum()} with a full ectodomain")
ecto_out.drop(columns="ecto_seq").head()

### 11b. Two-column table in the `Alleles_tab.txt` format

Same layout as `preMSA/Alleles_tab.txt`: tab-separated, header `Allele` / `seq`, one row per
sequence, and allele names carrying no `*` or `:` (so `KIR2DL1*001` becomes `KIR2DL1_001`, the
same way `HLA-A*01:01` becomes `HLA_A0101` there). The reference file ends without a trailing
newline and this one matches it.

In [ ]:
tab = (ecto_uniq.sort_values(["gene", "group3"])
       .assign(Allele=lambda d: d.gene + "_" + d.group3)
       .rename(columns={"ecto_seq": "seq"})[["Allele", "seq"]])

assert tab.Allele.is_unique, "duplicate allele names in the table"
assert not tab.seq.str.contains("-").any(), "gap character in an unaligned sequence"

tab_path = PROJ / "KIR_Alleles_tab.txt"
# written by hand rather than to_csv: the reference file has no trailing newline
tab_path.write_text(
    "Allele\tseq\n" + "\n".join(f"{a}\t{s}" for a, s in zip(tab.Allele, tab.seq))
)

print(f"wrote {tab_path.name}: {len(tab)} rows, {tab_path.stat().st_size / 1024:.0f} KB")
tab.head()

## Visualising the alignments

Two views of the same data: an inline difference table for working in the notebook, and a
Clustal-style coloured PDF per gene for reading or printing.

Residues are coloured by physicochemical class, never one hue per amino acid - twenty colours
cannot be told apart reliably. The **letter** always carries the identity, so the colour is a
grouping aid rather than the encoding. Residues matching the column consensus are greyed, and
substitutions are bold black, which is what makes the changes legible at a glance.

Positions are numbered in **mature-protein** coordinates (1 = first residue after the signal
peptide), the convention used in the KIR structural literature.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
import math

PDF_DIR = PROJ / "alignment_pdfs"
PDF_DIR.mkdir(exist_ok=True)

CLASS_COLOR = {
    "hydrophobic": "#bcd6ef",   # A I L M F W V
    "positive":    "#f3bdb6",   # K R
    "negative":    "#e8c4de",   # D E
    "polar":       "#bfe3cc",   # N Q S T
    "cysteine":    "#f6c9d8",   # C
    "glycine":     "#f7d9b0",   # G
    "proline":     "#f6ebb0",   # P
    "aromatic":    "#bfe3e3",   # H Y
}
RESIDUE_CLASS = {r: cls for cls, residues in [
    ("hydrophobic", "AILMFWV"), ("positive", "KR"), ("negative", "DE"),
    ("polar", "NQST"), ("cysteine", "C"), ("glycine", "G"),
    ("proline", "P"), ("aromatic", "HY"),
] for r in residues}

INK, MUTED, GRID = "#0b0b0b", "#6f6e6a", "#d8d7d2"
HILITE = "#d94f2b"

### 12. Which positions to highlight

**This is not a structural contact set.** It holds a single position: mature residue 44, the
classical KIR2DL1-vs-2DL2/3 specificity dimorphism, and the only highlighted position with
support outside this notebook.

It reproduces at exactly the expected place, which independently confirms the ectodomain
boundary - had the signal-peptide cut been off by one, the landmark would sit at 43 or 45. Note
that in this data the C2-binding genes carry **Met44** and the C1-binding genes **Lys44**, which
is consistent with HLA-C2 carrying Lys80 (a Lys44 would face a like charge).

Positions 150 and 190 were highlighted in an earlier version and have been removed. They came
from a scan for columns separating the C2- from the C1-binding genes, with no literature or
structural support, and separating by specificity is not the same thing as contacting HLA.

A real contact set should be computed from the co-crystals in `261105_kir_hla_structures.xlsx`
(1IM9, 1EFX, 6PA1, 6PAG, 3VH8, 6V3J, 3WUW, 7DUU, 4N8V) and dropped into `BINDING_POSITIONS`.

In [ ]:
# mature-protein residue numbers; replace with structure-derived contacts when available
BINDING_POSITIONS = {
    "KIR2DL1": [44], "KIR2DS1": [44],
    "KIR2DL2": [44], "KIR2DL3": [44], "KIR2DS2": [44],
}


def load_alignment(gene, kind="ecto"):
    """Labels and aligned strings for one gene."""
    recs = list(SeqIO.parse(OUT_DIR / f"{gene}_{kind}_aln.fasta", "fasta"))
    return [r.id.split("|")[0] for r in recs], [str(r.seq) for r in recs]


def column_consensus(seqs):
    out = []
    for i in range(len(seqs[0])):
        col = [s[i] for s in seqs if s[i] != "-"]
        out.append(max(set(col), key=col.count) if col else "-")
    return out

### 13. Inline difference table

Variable positions only, shown as differences from the consensus: a dot means "same as
consensus", a letter means a substitution. Alleles are ordered most-divergent first, so a
truncated view still shows the interesting ones.

In [ ]:
def alignment_diff(gene, kind="ecto", max_rows=None):
    """Variable columns as differences from the consensus, mature numbering."""
    labels, seqs = load_alignment(gene, kind)
    mat = pd.DataFrame([list(s) for s in seqs], index=labels,
                       columns=range(1, len(seqs[0]) + 1))

    n_resid = mat.apply(lambda c: c[c != "-"].nunique())
    var = mat.loc[:, n_resid > 1]
    consensus = var.apply(lambda c: c[c != "-"].mode().iat[0])
    diff = var.where(var.ne(consensus, axis=1), ".")

    order = (diff != ".").sum(axis=1).sort_values(ascending=False).index
    diff = diff.loc[order]
    if max_rows:
        diff = diff.head(max_rows)
    return pd.concat([consensus.to_frame("consensus").T, diff])


def style_diff(df, gene=None):
    """Grey dots for consensus, bold accent for substitutions."""
    hl = set(BINDING_POSITIONS.get(gene, [])) if gene else set()

    def cell(v):
        return f"color:{MUTED}" if v == "." else f"color:{HILITE};font-weight:700"

    styler = (df.style.map(cell)
              .set_table_styles([
                  {"selector": "th", "props": [("font-size", "10px"), ("color", INK)]},
                  {"selector": "td", "props": [("font-family", "monospace"),
                                               ("font-size", "11px"),
                                               ("text-align", "center"),
                                               ("padding", "1px 3px")]}]))
    if hl:
        cols = [c for c in df.columns if c in hl]
        if cols:
            styler = styler.set_properties(subset=cols,
                                           **{"background-color": "#fdece7"})
    return styler


style_diff(alignment_diff("KIR2DL2"), gene="KIR2DL2")

### 14. One Clustal-style PDF per gene

Wrapped at 60 columns per block, one block per page, with a residue ruler, a colour-class legend
on the first page, and the highlighted positions boxed and marked.

In [ ]:
def render_gene_pdf(gene, kind="ecto", block=60, highlight=None, out_dir=None):
    """Clustal-style coloured alignment PDF for one gene."""
    out_dir = out_dir or PDF_DIR
    labels, seqs = load_alignment(gene, kind)
    width = len(seqs[0])
    highlight = set(highlight if highlight is not None
                    else BINDING_POSITIONS.get(gene, []))
    consensus = column_consensus(seqs)

    cell_w, cell_h, label_w = 0.105, 0.135, 1.45
    path = out_dir / f"{gene}_alignment.pdf"

    with PdfPages(path) as pdf:
        for b in range(math.ceil(width / block)):
            lo, hi = b * block, min((b + 1) * block, width)
            ncol = hi - lo

            fig_w = label_w + ncol * cell_w + 0.3
            fig_h = 0.95 + len(seqs) * cell_h
            fig = plt.figure(figsize=(fig_w, fig_h))
            ax = fig.add_axes([label_w / fig_w, 0.1 / fig_h,
                               ncol * cell_w / fig_w, len(seqs) * cell_h / fig_h])
            ax.set_xlim(0, ncol)
            ax.set_ylim(len(seqs), 0)
            ax.axis("off")

            for r, (lab, seq) in enumerate(zip(labels, seqs)):
                for c in range(lo, hi):
                    aa, x = seq[c], c - lo
                    cls = RESIDUE_CLASS.get(aa)
                    if cls:
                        ax.add_patch(Rectangle((x, r), 1, 1, linewidth=0,
                                               facecolor=CLASS_COLOR[cls]))
                    same = aa == consensus[c]
                    ax.text(x + 0.5, r + 0.5, aa, ha="center", va="center",
                            fontsize=5.6, family="monospace",
                            color=MUTED if same else INK,
                            weight="normal" if same else "bold")
                ax.text(-0.4, r + 0.5, lab, ha="right", va="center",
                        fontsize=5.6, family="monospace", color=INK)

            for pos in sorted(highlight):
                if lo < pos <= hi:
                    x = pos - 1 - lo
                    ax.add_patch(Rectangle((x, 0), 1, len(seqs), linewidth=0.9,
                                           edgecolor=HILITE, facecolor="none", zorder=5))
                    ax.text(x + 0.5, -0.45, "\u25bc", ha="center", va="center",
                            fontsize=4.5, color=HILITE, clip_on=False)

            for c in range(lo, hi):
                if (c + 1) % 10 == 0:
                    ax.text(c - lo + 0.5, len(seqs) + 0.55, str(c + 1),
                            ha="center", va="center", fontsize=4.8,
                            color=MUTED, clip_on=False)

            fig.text(0.012, 1 - 0.3 / fig_h,
                     f"{gene}  -  {kind}domain alignment  "
                     f"({len(seqs)} unique, residues {lo + 1}-{hi} of {width})",
                     fontsize=8, color=INK, va="top")

            if b == 0:
                x = 0.012
                for cls, col in CLASS_COLOR.items():
                    fig.patches.append(plt.Rectangle(
                        (x, 1 - 0.72 / fig_h), 0.012, 0.16 / fig_h,
                        transform=fig.transFigure, facecolor=col, linewidth=0))
                    fig.text(x + 0.016, 1 - 0.64 / fig_h, cls, fontsize=5.2,
                             color=MUTED, va="center")
                    x += 0.016 + 0.011 * len(cls)
                if highlight:
                    # own line: a trailing entry clips off narrow pages
                    fig.text(0.012, 1 - 0.80 / fig_h,
                             "\u25bc residue 44 - C1/C2 specificity dimorphism",
                             fontsize=5.2, color=HILITE, va="center")

            pdf.savefig(fig)
            plt.close(fig)

    return path

In [ ]:
pdf_paths = {g: render_gene_pdf(g) for g in sorted(ecto_aln_paths)}  # ~45 s
for g, p in pdf_paths.items():
    print(f"  {g:9s} {p.name:28s} {p.stat().st_size / 1024:6.0f} KB")